In [1]:
# Cell 1 — parameters

LAT        = 16.8167
LON        = -2.9833
LEVEL      = 6
FROM_YEAR  = 1100
TO_YEAR    = 1200
HYBAS_ID   = 1060551560          # confirmed: both engines, both levels, same basin

BASE_URL   = 'http://localhost:8000'

In [2]:
# Cell 2 — imports and output path

import sys, json
from pathlib import Path
import requests

sys.path.insert(0, str(Path('.').resolve()))
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).resolve().parents[2]
OUT  = ROOT / 'output' / 'edop' / 'areas'

In [3]:
# Cell 3 — fetch v0.3 signature and save to disk

url = (f'{BASE_URL}/api/signature'
       f'?lat={LAT}&lon={LON}'
       f'&bands=ABCDET&level={LEVEL}'
       f'&from_year={FROM_YEAR}&to_year={TO_YEAR}')

resp = requests.get(url)
resp.raise_for_status()
v3 = resp.json()

out_path = OUT / 'v03_timbuktu_signature.json'
out_path.write_text(json.dumps(v3, indent=2))

print(f'Status: {resp.status_code}')
print(f'Top-level keys: {list(v3.keys())}')
print(f'Bands present: {list(v3["profile_groups"].keys())}')
print(f'Basin id (v0.3 internal): {v3["id"]}')
print()

# Bands A–E use items list
for band in ['A', 'B', 'C', 'D', 'E']:
    grp   = v3['profile_groups'][band]
    items = grp.get('items', [])
    print(f'  Band {band}: {len(items)} items — {[i["key"] for i in items]}')

# Band T: LMR uses a single grid cell; HYDE aggregates n_cells within the basin
t    = v3['profile_groups']['T']
hyde = t.get('hyde_land_use') or {}
epochs = hyde.get('epochs') or []

print()
print(f'  Band T (LMR): grid_cell={t["grid_cell"]}  ({len(t.get("pdsi_series") or [])} annual values)')
print(f'    pdsi  mean={t["pdsi_mean"]}  min={t["pdsi_min"]}  max={t["pdsi_max"]}')
print(f'    temp  mean_anom_k={t["air_mean_anom_k"]}')
print(f'    precip mean_anom_mm_day={t["prate_mean_anom_mm_day"]}')
print(f'    volcanic_events={len(t.get("volcanic_events") or [])}')
print()
print(f'  Band T (HYDE): {len(epochs)} epoch snapshots (span endpoints only — not full series)')
for ep in epochs:
    print(f'    year={ep["year_ce"]}  n_cells={ep["n_cells"]}  basin_area={ep["basin_area_km2"]} km²')
    print(f'      cropland: {ep["cropland_km2"]} km² ({ep["cropland_pct"]}%)  '
          f'p10={ep["cropland_p10"]}  p90={ep["cropland_p90"]}  std={ep["cropland_std"]}')
    print(f'      grazing:  {ep["grazing_km2"]} km² ({ep["grazing_pct"]}%)  '
          f'p10={ep["grazing_p10"]}  p90={ep["grazing_p90"]}  std={ep["grazing_std"]}')

print()
print(f'Saved → {out_path}')

Status: 200
Top-level keys: ['id', 'eco_id', 'up_area', 'geom_geojson', 'elev_point', 'elev_source', 'elev_dataset', 'elev_resolution_m', 'relief_range_m', 'relief_position', 'profile_summary', 'profile_groups', 'meta']
Bands present: ['A', 'B', 'C', 'D', 'E', 'T']
Basin id (v0.3 internal): 1478

  Band A: 8 items — ['elev_min', 'elev_max', 'slope_avg', 'slope_upstream', 'stream_gradient', 'lith_class', 'karst', 'karst_upstream']
  Band B: 20 items — ['runoff', 'discharge_yr', 'discharge_min', 'discharge_max', 'river_area', 'river_area_upstream', 'gw_table_depth', 'pnv_majority', 'pnv_shares', 'pct_clay', 'pct_silt', 'pct_sand', 'pct_clay_upstream', 'pct_silt_upstream', 'pct_sand_upstream', 'wet_pct_grp1', 'wet_pct_grp2', 'wet_pct_grp1_upstream', 'wet_pct_grp2_upstream', 'wetland_class']
  Band C: 13 items — ['temp_yr', 'temp_min', 'temp_max', 'temp_yr_upstream', 'precip_yr', 'precip_yr_upstream', 'aridity', 'aridity_upstream', 'permafrost_extent', 'biome', 'ecoregion', 'freshwater_eco

In [4]:
# Cell 4 — resolver sign-off gate (run before full payload)
# Report the one-entry weighted set: hybas_id, area, weight, shortfall.
# Expected: hybas_id=1060551560, weight=1.0, shortfall=0.0

import sys, warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy', category=UserWarning)
sys.path.insert(0, str(Path('.').resolve()))

from scripts.shared.db_utils import db_connect
from scripts.edop.areas.engine import resolve_single_basin

conn = db_connect()
try:
    basin_set = resolve_single_basin(LAT, LON, LEVEL, conn)
finally:
    conn.close()

assert len(basin_set) == 1,        f'FAIL: expected 1 basin, got {len(basin_set)}'
assert basin_set['weight'].iloc[0] == 1.0, 'FAIL: weight must be 1.0'
assert int(basin_set['hybas_id'].iloc[0]) == HYBAS_ID, \
    f'FAIL: hybas_id={basin_set["hybas_id"].iloc[0]}, expected {HYBAS_ID}'

print('Resolver output:')
print(basin_set.to_string(index=False))
print()
print(f'shortfall = 0.0  (structural: query IS the basin)')
print(f'PASS — hybas_id={HYBAS_ID}, weight=1.0')

Resolver output:
  hybas_id  weight
1060551560     1.0

shortfall = 0.0  (structural: query IS the basin)
PASS — hybas_id=1060551560, weight=1.0


In [5]:
# Cell 5 — run v0.4 single_basin_signature and persist payload

from scripts.edop.areas.engine import single_basin_signature

conn = db_connect()
try:
    v4 = single_basin_signature(LAT, LON, conn, level=LEVEL, include_detail=True)
finally:
    conn.close()

out_path_v4 = OUT / 'v04_timbuktu_single_basin.json'
out_path_v4.write_text(json.dumps(v4, indent=2))

nb = v4['neighborhood']
rows = v4['rows']

print(f'neighborhood: type={nb["type"]}  hybas_id={nb["hybas_id"]}  '
      f'n_units={nb["n_units"]}  level={nb["level"]}')
print(f'shortfall: {v4["shortfall"]}')
print(f'bands: {v4["bands"]}')
print(f'rows: {len(rows)}')
print()

# Quick inventory by method
from collections import Counter
by_method = Counter(r['method'] for r in rows)
for method, n in sorted(by_method.items()):
    print(f'  {method:<20} {n} rows')

print()
print(f'Saved → {out_path_v4}')

neighborhood: type=basin  hybas_id=1060551560  n_units=1  level=6
shortfall: 0.0
bands: ['A', 'B', 'C', 'D', 'E']
rows: 51

  area_weighted        34 rows
  class_mixture        10 rows
  distribution_only    2 rows
  dominant_basin       3 rows
  extreme              1 rows
  flag_fraction        1 rows

Saved → /Users/karlg/Documents/repos/_edops/output/edop/areas/v04_timbuktu_single_basin.json
